# SIGMOD Exp2 Frozen-Trace Crossover

This notebook fixes a single randomized base trace and rewrites only selected operations for each sweep point:
- history sweep: `RecentScan -> HistoryScan`, then `RecentProbe -> HistoricalProbe`
- delta sweep: `RecentScan -> DeltaScan`

The goal is to compare apples-to-apples at `0%, 1%, 2%, 4%, 6%, 8%, 10%`.

In [ ]:
from pathlib import Path
import sys
import json
import random
import importlib
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_HTAP_TXN_COUNT,
    SIGMOD_HTAP_WAREHOUSE_COUNT,
    SIGMOD_READABLE_EVERY,
    apply_paper_style,
    current_run_stamp,
    ensure_dirs,
    run_checked,
)
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp2_frozen_crossover').resolve()
DATA_DIR = (ROOT / 'benches' / 'sigmod_data').resolve()
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TABLE_TYPES = ['ivmh', 'heap', 'chain', 'par']
TX_MAP = {'MarkTs': 'BuildSnap', 'DelSc': 'DeltaScan'}

CONFIG = {
    'warehouse_count': SIGMOD_HTAP_WAREHOUSE_COUNT,
    'txn_count': SIGMOD_HTAP_TXN_COUNT,
    'bucket_num': SIGMOD_BUCKET_NUM,
    'update_ratio': 0.0001,
    'probe_ratio': 0.01,
    'txn_gc_ratio': 0.05,
    'readable_every': SIGMOD_READABLE_EVERY,
    'repeat': 5,
    'trim': 1,
    'timeout_sec': 900,
    'rerun_snap': False,
    'trace_seed': 232323223,
}

SWEEP = {
    'history_values': [0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.10],
    'delta_values': [0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.10],
}

STYLE = {
    ('naive', ''): ('SNAP', TOL['red'], ':', 'x'),
    ('ivmh', ''): ('IVMH', TOL['yellow'], '--', 'P'),
    ('heap', 'Write Repair'): ('MONO-WR', TOL['blue'], '-', 'o'),
    ('chain', 'Write Repair'): ('DUAL-WR', TOL['cyan'], '-', 's'),
    ('par', 'Write Repair'): ('EPOCH-WR', TOL['green'], '-', 'D'),
}

RUN_STAMP = current_run_stamp()

CONFIG_TAG = '_'.join([
    f"wc{CONFIG['warehouse_count']}",
    f"tc{CONFIG['txn_count']}",
    f"bn{CONFIG['bucket_num']}",
    f"ur{str(CONFIG['update_ratio']).replace('.', 'p')}",
    f"pr{str(CONFIG['probe_ratio']).replace('.', 'p')}",
    f"gc{str(CONFIG['txn_gc_ratio']).replace('.', 'p')}",
    f"re{CONFIG['readable_every']}",
    f"rep{CONFIG['repeat']}",
    f"seed{CONFIG['trace_seed']}",
    'frozen',
])

BASE_ARGS = [
    '--txn-count', str(CONFIG['txn_count']),
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--update-ratio', str(CONFIG['update_ratio']),
    '--probe-ratio', str(CONFIG['probe_ratio']),
    '--bucket-num', str(CONFIG['bucket_num']),
    '--readable-every', str(CONFIG['readable_every']),
    '--seed', str(CONFIG['trace_seed']),
]

print('ROOT   :', ROOT)
print('BIN    :', BIN)
print('OUTDIR :', DATA_DIR)
print('CONFIG :', CONFIG)
print('STAMP  :', RUN_STAMP)
print('TAG    :', CONFIG_TAG)


In [ ]:
FIXED_SNAP_DELTA_ROWS = [
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 40139.22504744444, 'total_ms': 401.3922504744444, 'delta_ratio': 0.00},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 41288.46775166667, 'total_ms': 412.8846775166667, 'delta_ratio': 0.01},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 41559.591691444446, 'total_ms': 415.5959169144445, 'delta_ratio': 0.02},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 42129.533485333326, 'total_ms': 421.29533485333326, 'delta_ratio': 0.04},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 42632.21794944445, 'total_ms': 426.3221794944445, 'delta_ratio': 0.06},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 42890.86272711111, 'total_ms': 428.90862727111113, 'delta_ratio': 0.08},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 42336.72021111111, 'total_ms': 423.3672021111111, 'delta_ratio': 0.10},
]

FIXED_SNAP_HISTORY_ROWS = [
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 40651.66512711111, 'total_ms': 406.51665127111113, 'history_ratio': 0.00},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 40687.81570233333, 'total_ms': 406.87815702333336, 'history_ratio': 0.01},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 39695.93911433333, 'total_ms': 396.9593911433333, 'history_ratio': 0.02},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 39469.83057122222, 'total_ms': 394.69830571222224, 'history_ratio': 0.04},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 39229.225423, 'total_ms': 392.29225423, 'history_ratio': 0.06},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 38858.987995888885, 'total_ms': 388.5898799588888, 'history_ratio': 0.08},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 37854.40020022222, 'total_ms': 378.5440020022222, 'history_ratio': 0.10},
]

def merge_args(base, extra):
    return [*base, *extra]

def trim_trial_runs(df):
    trim = CONFIG['trim']
    if trim <= 0 or df.empty or 'trial' not in df.columns:
        return df
    totals = (
        df.groupby('trial', as_index=False)['duration_ms']
        .sum()
        .sort_values('duration_ms')
    )
    if len(totals) <= 2 * trim:
        return df
    keep = set(totals.iloc[trim:len(totals) - trim]['trial'])
    return df[df['trial'].isin(keep)].copy()

def collapse_repairs(df, table_type):
    if table_type in {'naive', 'ivmh'}:
        collapsed = df.groupby(['table_type'], as_index=False)['duration_ms'].mean()
        collapsed['repair_type'] = ''
        return collapsed[['table_type', 'repair_type', 'duration_ms']]
    return df

def scaled_tx_counts(update_share, probe_share, scan_share, delta_share):
    analytical_and_update = 1.0 - CONFIG['txn_gc_ratio']
    update_n = round(CONFIG['txn_count'] * analytical_and_update * update_share)
    probe_n = round(CONFIG['txn_count'] * analytical_and_update * probe_share)
    delta_n = round(CONFIG['txn_count'] * analytical_and_update * delta_share)
    gc_n = round(CONFIG['txn_count'] * CONFIG['txn_gc_ratio'])
    scan_n = CONFIG['txn_count'] - update_n - probe_n - delta_n - gc_n
    return update_n, probe_n, scan_n, delta_n, gc_n

def build_base_sequence():
    update_n, probe_n, scan_n, delta_n, gc_n = scaled_tx_counts(0.20, 0.40, 0.40, 0.0)
    assert delta_n == 0
    seq = ['U'] * update_n + ['P'] * probe_n + ['R'] * scan_n + ['G'] * gc_n
    rng = random.Random(CONFIG['trace_seed'])
    rng.shuffle(seq)
    return seq

BASE_SEQUENCE = build_base_sequence()
print('Base sequence length:', len(BASE_SEQUENCE))
print('Base counts:', {ch: BASE_SEQUENCE.count(ch) for ch in ['U','P','R','G']})

def rewrite_history_sequence(history_pct):
    total_history_ops = int(round(CONFIG['txn_count'] * history_pct))
    seq = BASE_SEQUENCE.copy()
    if total_history_ops <= 1:
        history_scan_count = total_history_ops
        history_probe_count = 0
    else:
        history_probe_count = total_history_ops // 2
        history_scan_count = total_history_ops - history_probe_count
    scan_slots = [i for i, op in enumerate(seq) if op == 'R']
    probe_slots = [i for i, op in enumerate(seq) if op == 'P']
    for idx in scan_slots[:history_scan_count]:
        seq[idx] = 'S'
    for idx in probe_slots[:history_probe_count]:
        seq[idx] = 'H'
    return seq

def rewrite_delta_sequence(delta_pct):
    total_delta_ops = int(round(CONFIG['txn_count'] * delta_pct))
    seq = BASE_SEQUENCE.copy()
    scan_slots = [i for i, op in enumerate(seq) if op == 'R']
    for idx in scan_slots[:total_delta_ops]:
        seq[idx] = 'D'
    return seq

def mix_from_sequence(seq):
    update_n = sum(op == 'U' for op in seq)
    probe_n = sum(op in {'P', 'H'} for op in seq)
    scan_n = sum(op in {'R', 'S'} for op in seq)
    delta_n = sum(op == 'D' for op in seq)
    gc_n = sum(op == 'G' for op in seq)
    non_gc = update_n + probe_n + scan_n + delta_n
    return {
        'update_ratio': update_n / non_gc,
        'probe_ratio': probe_n / non_gc,
        'scan_ratio': scan_n / non_gc,
        'delta_ratio': delta_n / non_gc,
        'gc_count': gc_n,
    }

def build_manual_args(seq):
    mix = mix_from_sequence(seq)
    args = [
        '--txn-update-ratio', str(mix['update_ratio']),
        '--txn-probe-ratio', str(mix['probe_ratio']),
        '--txn-scan-ratio', str(mix['scan_ratio']),
        '--txn-delta-ratio', str(mix['delta_ratio']),
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
        '--scan-reuse-ratio', '0.0',
        '--probe-history-ratio', '0.0',
        '--manual-txs', ''.join(seq),
    ]
    if any(op in {'H', 'S'} for op in seq):
        args.append('--distinct-history-targets')
    if 'D' in seq:
        args.append('--distinct-delta-targets')
    return args

def run_single_table(extra_args, table_type):
    trials = []
    print('  table=', table_type)
    for trial in range(CONFIG['repeat']):
        result = run_checked([str(BIN), *merge_args(BASE_ARGS, extra_args), '--table-type', table_type], ROOT, quiet=True, timeout=CONFIG['timeout_sec'])
        df = parse_result(result.stdout, table_type)
        if df.empty:
            raise RuntimeError(f'No parsed rows for {table_type}')
        df['tx_type'] = df['tx_type'].replace(TX_MAP)
        total = df.groupby('repair_type', as_index=False)['duration_ms'].sum()
        total['table_type'] = table_type
        total['trial'] = trial
        trials.append(total)
    df_all = trim_trial_runs(pd.concat(trials, ignore_index=True))
    df_avg = df_all.groupby(['table_type', 'repair_type'], as_index=False)['duration_ms'].mean()
    out = collapse_repairs(df_avg, table_type)
    out['total_ms'] = out['duration_ms'] / CONFIG['txn_count']
    return out

def normalize_sweep_value(value):
    return round(float(value), 10)

def stable_csv_path(stem):
    return DATA_DIR / f'{stem}_{CONFIG_TAG}.csv'

def dedup_result_rows(df, x_col):
    if df.empty:
        return df
    out = df.copy()
    out['repair_type'] = out['repair_type'].fillna('')
    out[x_col] = out[x_col].astype(float).map(normalize_sweep_value)
    out = out.drop_duplicates(subset=[x_col, 'table_type', 'repair_type'], keep='last')
    return out.sort_values([x_col, 'table_type', 'repair_type']).reset_index(drop=True)

def add_snap_rows(df, fixed_rows, x_col):
    snap_df = pd.DataFrame(fixed_rows)
    out = pd.concat([snap_df, df], ignore_index=True)
    out['repair_type'] = out['repair_type'].fillna('')
    return dedup_result_rows(out, x_col)

def fresh_sweep_df(fixed_rows, x_col):
    if CONFIG['rerun_snap']:
        return pd.DataFrame()
    return add_snap_rows(pd.DataFrame(), fixed_rows, x_col)

def upsert_table_rows(df, table_df, x_col):
    table_type = table_df['table_type'].iloc[0]
    value = normalize_sweep_value(table_df[x_col].iloc[0])
    if df.empty:
        return dedup_result_rows(table_df, x_col)
    out = df[~((df['table_type'] == table_type) & (df[x_col].astype(float).map(normalize_sweep_value) == value))].copy()
    out = pd.concat([out, table_df], ignore_index=True)
    return dedup_result_rows(out, x_col)

def _snap_break_bounds(df):
    snap = df[df['table_type'] == 'naive']['total_ms']
    others = df[df['table_type'] != 'naive']['total_ms']
    if snap.empty or others.empty:
        return None
    lower_max = others.max() * 1.08
    upper_min = snap.min() * 0.92
    if upper_min <= lower_max:
        midpoint = (others.max() + snap.min()) / 2.0
        lower_max = midpoint * 0.95
        upper_min = midpoint * 1.05
    upper_max = snap.max() * 1.03
    return lower_max, upper_min, upper_max

def plot_one(ax, df, x_col, xlabel):
    series = [('naive', ''), ('ivmh', ''), ('heap', 'Write Repair'), ('chain', 'Write Repair'), ('par', 'Write Repair')]
    for key in series:
        label, color, linestyle, marker = STYLE[key]
        table_type, repair_type = key
        sub = df[(df['table_type'] == table_type) & (df['repair_type'] == repair_type)].sort_values(x_col)
        if sub.empty:
            continue
        ax.plot(sub[x_col], sub['total_ms'], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Duration (ms / tx)')
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    ax.set_ylim(bottom=0)
    ax.legend(loc='center left', ncol=1, framealpha=0.95)

def _save_fig(fig, stem):
    out_pdf = FIGS_DIR / f'{stem}_{RUN_STAMP}.pdf'
    latest_pdf = FIGS_DIR / f'{stem}.pdf'
    fig.savefig(out_pdf, format='pdf')
    fig.savefig(latest_pdf, format='pdf')
    plt.show()
    print('Saved', out_pdf)
    print('Saved', latest_pdf)

def render_single_broken_snap(df, x_col, xlabel, stem):
    bounds = _snap_break_bounds(df)
    fig, (ax_top, ax_bottom) = plt.subplots(2, 1, figsize=(5.0, 4.6), sharex=True, gridspec_kw={'height_ratios': [1.0, 3.0], 'hspace': 0.05})
    plot_one(ax_top, df, x_col, '')
    plot_one(ax_bottom, df, x_col, xlabel)
    if bounds is not None:
        lower_max, upper_min, upper_max = bounds
        ax_top.set_ylim(upper_min, upper_max)
        ax_bottom.set_ylim(0, lower_max)
        ax_top.spines['bottom'].set_visible(False)
        ax_bottom.spines['top'].set_visible(False)
        ax_top.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
        ax_top.set_xlabel('')
        ax_top.set_ylabel('')
        if ax_bottom.legend_ is not None:
            ax_bottom.legend_.remove()
        d = 0.012
        kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False, linewidth=0.8)
        ax_top.plot((-d, +d), (-d, +d), **kwargs)
        ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)
        kwargs.update(transform=ax_bottom.transAxes)
        ax_bottom.plot((-d, +d), (1 - d, 1 + d), **kwargs)
        ax_bottom.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)
    fig.tight_layout()
    _save_fig(fig, stem)

def run_history_sweep():
    csv_path = stable_csv_path('sigmod_exp2_frozen_history')
    df = fresh_sweep_df(FIXED_SNAP_HISTORY_ROWS, 'history_ratio')
    df.to_csv(csv_path, index=False)
    for value in SWEEP['history_values']:
        print(f'Running history sweep at {value:.2%}')
        seq = rewrite_history_sequence(value)
        args = build_manual_args(seq)
        for table_type in TABLE_TYPES:
            table_df = run_single_table(args, table_type)
            table_df['history_ratio'] = normalize_sweep_value(value)
            df = upsert_table_rows(df, table_df, 'history_ratio')
            df.to_csv(csv_path, index=False)
            print('  saved', csv_path)
    df = dedup_result_rows(df, 'history_ratio')
    df.to_csv(csv_path, index=False)
    return df, csv_path

def run_delta_sweep():
    csv_path = stable_csv_path('sigmod_exp2_frozen_delta')
    df = fresh_sweep_df(FIXED_SNAP_DELTA_ROWS, 'delta_ratio')
    df.to_csv(csv_path, index=False)
    for value in SWEEP['delta_values']:
        print(f'Running delta sweep at {value:.2%}')
        seq = rewrite_delta_sequence(value)
        args = build_manual_args(seq)
        for table_type in TABLE_TYPES:
            table_df = run_single_table(args, table_type)
            table_df['delta_ratio'] = normalize_sweep_value(value)
            df = upsert_table_rows(df, table_df, 'delta_ratio')
            df.to_csv(csv_path, index=False)
            print('  saved', csv_path)
    df = dedup_result_rows(df, 'delta_ratio')
    df.to_csv(csv_path, index=False)
    return df, csv_path


In [ ]:
df_history, history_csv = run_history_sweep()
df_history['history_pct'] = df_history['history_ratio'] * 100.0
print('History CSV:', history_csv)
render_single_broken_snap(df_history, 'history_pct', 'Historical Operation Percentage (%)', 'exp2-frozen-history-with-snap-broken')


In [ ]:
df_delta, delta_csv = run_delta_sweep()
df_delta['delta_pct'] = df_delta['delta_ratio'] * 100.0
print('Delta CSV:', delta_csv)
render_single_broken_snap(df_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'exp2-frozen-delta-with-snap-broken')
